# ES|QL Complete Reference Guide
### Built from hands-on practice with `kibana_sample_data_logs`

---

## What is ES|QL?

ES|QL (Elasticsearch Query Language) is a **pipeline-based query language** introduced in Elasticsearch 8.11. Unlike SQL, it does not use SELECT/FROM/WHERE in one block. Instead, it chains commands with a pipe `|` — each command transforms the output of the previous one.

```
SOURCE | TRANSFORM | TRANSFORM | ... | OUTPUT
```

Every query starts with a source (`FROM`), then you shape, filter, compute, aggregate, and sort — in that order, one pipe at a time.

---

## The Three Kibana Layers — Know Which One to Use

```
┌──────────────────────────────────────────────────┐
│  LAYER 1 — Time picker       (top bar)           │
│  Shard-level pruning. Cheapest. Always first.    │
├──────────────────────────────────────────────────┤
│  LAYER 2 — DSL filter bar    (≡ button)          │
│  Runs before ES|QL sees data. Skips shards.      │
│  Best for: keyword fields, exact known values.   │
├──────────────────────────────────────────────────┤
│  LAYER 3 — ES|QL pipeline    (search bar)        │
│  Row-level transforms after data is loaded.      │
│  Best for: compute, aggregate, shape output.     │
└──────────────────────────────────────────────────┘
```

**Rule of thumb**: filter first with the time picker and DSL bar, compute with ES|QL.

---

## Command Reference

### FROM — the source

Every query starts here. Selects which index or data stream to read from.

```esql
FROM kibana_sample_data_logs          -- single index
FROM logs-*, metrics-*                -- wildcard + multiple patterns
FROM logs-* METADATA _index, _id      -- include ES internal fields
```

---

### STATS — aggregate and group

The most powerful command. Computes aggregations, optionally grouped by a field.

```esql
-- Total count
FROM kibana_sample_data_logs
| STATS total = COUNT(*)

-- Count grouped by a field
FROM kibana_sample_data_logs
| STATS count = COUNT(*) BY response
| SORT count DESC
```

**Real result from your dataset:**
```
count  | response
-------|----------
12832  | "200"
801    | "404"
441    | "503"
```

**Multiple aggregations at once:**
```esql
FROM kibana_sample_data_logs
| STATS
    requests  = COUNT(*),
    avg_bytes = ROUND(AVG(bytes), 0),
    p95_bytes = PERCENTILE(bytes, 95),
    max_bytes = MAX(bytes)
  BY extension.keyword
| SORT avg_bytes DESC
```

**Real result:**
```
extension | requests | avg_bytes | p95_bytes | max_bytes
----------|----------|-----------|-----------|----------
deb       | 1605     | 6346      | 12399     | 19904
zip       | 1674     | 6247      | 12101     | 19856
rpm       | 567      | 6181      | 13624     | 19711
gz        | 2642     | 6148      | 11286     | 19929
css       | 2255     | 5589      | 9574      | 9998
```

**Insight**: `.deb` and `.zip` files are the heaviest downloads. `css` has a much lower max (9998 bytes) because CSS files are capped in size — no large binary content.

**Available aggregation functions:**

| Function | What it does | Example |
|---|---|---|
| `COUNT(*)` | Total row count | `COUNT(*)` |
| `COUNT_DISTINCT(field)` | Unique values | `COUNT_DISTINCT(clientip)` |
| `SUM(field)` | Total | `SUM(bytes)` |
| `AVG(field)` | Mean | `AVG(bytes)` |
| `MIN(field)` | Smallest value | `MIN(bytes)` |
| `MAX(field)` | Largest value | `MAX(bytes)` |
| `PERCENTILE(field, n)` | nth percentile | `PERCENTILE(bytes, 95)` |
| `MEDIAN(field)` | 50th percentile shorthand | `MEDIAN(bytes)` |

---

### EVAL — compute new columns

Creates a new column on every row. Runs before STATS, so you can aggregate on computed values.

```esql
-- Arithmetic
FROM kibana_sample_data_logs
| EVAL kb = ROUND(bytes / 1024.0, 2)
| STATS avg_kb = ROUND(AVG(kb), 2), max_kb = ROUND(MAX(kb), 2)
  BY extension.keyword
| SORT avg_kb DESC
```

**Real result:**
```
avg_kb | max_kb | extension
-------|--------|----------
6.2    | 19.44  | deb
6.1    | 19.39  | zip
6.04   | 19.25  | rpm
6.0    | 19.46  | gz
5.46   | 9.76   | css
4.9    | 19.52  | (empty)
```

**EVAL with CASE — if/then/else logic:**
```esql
FROM kibana_sample_data_logs
| EVAL size_band = CASE(
    bytes < 1000,   "small  < 1kb",
    bytes < 10000,  "medium 1–10kb",
    bytes < 100000, "large  10–100kb",
                    "huge   > 100kb"
  )
| STATS count = COUNT(*) BY size_band
| SORT count DESC
```

**Real result:**
```
count  | size_band
-------|------------------
12036  | medium 1–10kb
1336   | small < 1kb
702    | large 10–100kb
0      | huge > 100kb      ← no response exceeded 100kb in this dataset
```

**Insight**: This dataset is entirely small files — max response was ~20KB. No large binary payloads at all. In real web logs you would see the "huge" bucket for video or installer downloads.

**EVAL with type conversion — the `response` workaround:**

Because `response` is mapped as `text` (a mapping mistake), you cannot compare it numerically. The workaround:

```esql
FROM kibana_sample_data_logs
| EVAL is_error = CASE(TO_INTEGER(response.keyword) >= 500, 1, 0)
| STATS
    total     = COUNT(*),
    errors    = SUM(is_error),
    error_pct = ROUND(100.0 * SUM(is_error) / COUNT(*), 2)
```

**Real result:**
```
total  | errors | error_pct
-------|--------|----------
14074  | 441    | 3.13
```

**Insight**: 3.13% error rate. All 441 errors are `503` responses. Zero `500` errors — this dataset only contains `200`, `404`, and `503`.

> **Why `TO_INTEGER` is needed here**: `response` was stored as a string (`"200"`, `"404"`) because the source data sent it as text. ES|QL cannot compare text to an integer with `>=`. `TO_INTEGER()` converts the string to a number at query time. In a proper integer mapping this EVAL line is unnecessary — you write `WHERE response >= 500` directly.

---

### WHERE — filter rows

Filters rows after they are loaded. Use this for computed/EVAL'd fields or complex expressions. Use the DSL filter bar for indexed keyword fields — it is faster.

```esql
-- Exact match
FROM kibana_sample_data_logs
| WHERE response.keyword == "503"

-- Multiple conditions
FROM kibana_sample_data_logs
| WHERE response.keyword == "503" OR response.keyword == "404"

-- Null check
FROM kibana_sample_data_logs
| WHERE bytes IS NOT NULL

-- LIKE wildcard — % = any characters
FROM kibana_sample_data_logs
| WHERE request LIKE "%elasticsearch%"

-- RLIKE — regular expression
FROM kibana_sample_data_logs
| WHERE request RLIKE ".*elasticsearch-[0-9]+.*"

-- Numeric range
FROM kibana_sample_data_logs
| WHERE bytes > 50000
```

> **Important gotcha**: The query `WHERE bytes > 50000` against this dataset returns 0 rows — the sample data's max response is ~20KB. This is not a bug, it is the data. Always check your range assumptions against `MAX(field)` first.

---

### DATE_TRUNC — bucket by time

Groups timestamps into equal time buckets. Essential for trend charts.

```esql
FROM kibana_sample_data_logs
| STATS requests = COUNT(*)
    BY hour = DATE_TRUNC(1 hour, @timestamp)
| SORT hour ASC
| LIMIT 20
```

**Real result (first 8 hours of data):**
```
requests | hour
---------|------------------------
2        | 2026-03-15T00:00:00Z
12       | 2026-03-15T03:00:00Z
2        | 2026-03-15T04:00:00Z
6        | 2026-03-15T05:00:00Z
20       | 2026-03-15T06:00:00Z
15       | 2026-03-15T07:00:00Z
15       | 2026-03-15T08:00:00Z
16       | 2026-03-15T09:00:00Z
```

**Insight**: Traffic ramps up from ~06:00 UTC. The 03:00 spike (12 requests vs 2 at midnight) is worth investigating — could be a scheduled job or a bot.

**Time bucket sizes:**

| Bucket | Syntax | Use case |
|---|---|---|
| 1 minute | `1 minute` | Real-time alerting |
| 15 minutes | `15 minutes` | Incident investigation |
| 1 hour | `1 hour` | Daily traffic patterns |
| 1 day | `1 day` | Weekly trends |
| 1 week | `1 week` | Monthly overview |

---

### DATE_FORMAT — extract time parts

Extracts human-readable parts of a timestamp as a string.

```esql
FROM kibana_sample_data_logs
| EVAL day = DATE_FORMAT("EEEE", @timestamp)
| STATS requests = COUNT(*) BY day
| SORT requests DESC
```

**Real result:**
```
requests | day
---------|----------
2091     | Sunday
2076     | Wednesday
2072     | Monday
2046     | Thursday
2012     | Tuesday
1937     | Saturday
1840     | Friday
```

**Insight**: Traffic is remarkably even across all days — this is synthetic sample data. Real production logs typically show clear weekday/weekend splits. Sunday being highest here is a sample data artefact.

**Format patterns:**

| Pattern | Output | Use case |
|---|---|---|
| `EEEE` | `Monday` | Day of week analysis |
| `HH` | `14` | Hour of day (0–23) |
| `MM` | `03` | Month number |
| `yyyy` | `2026` | Year |
| `yyyy-MM-dd` | `2026-03-15` | Date-only grouping |

---

### KEEP / DROP / RENAME — shape output columns

Controls which columns appear in results and what they are called.

```esql
-- Keep only what you need (reduces output noise)
FROM kibana_sample_data_logs
| WHERE bytes > 10000
| KEEP @timestamp, request, bytes, response.keyword, clientip
| SORT bytes DESC
| LIMIT 20

-- Drop columns you do not need
FROM kibana_sample_data_logs
| STATS total_bytes = SUM(bytes), requests = COUNT(*) BY geo.srcdest
| DROP requests
| SORT total_bytes DESC

-- Rename for readability
FROM kibana_sample_data_logs
| STATS req = COUNT(*), bytes = SUM(bytes) BY geo.src
| RENAME geo.src AS country, req AS requests, bytes AS total_bytes
| EVAL total_mb = ROUND(total_bytes / 1048576.0, 2)
| DROP total_bytes
| SORT requests DESC
| LIMIT 10
```

---

### SORT / LIMIT — order and cap results

```esql
-- Single field sort
FROM kibana_sample_data_logs
| SORT bytes DESC
| LIMIT 10

-- Multi-field sort
FROM kibana_sample_data_logs
| STATS count = COUNT(*) BY response.keyword, host.keyword
| SORT count DESC, response.keyword ASC
| LIMIT 20
```

> **Rule**: Always put `SORT` and `LIMIT` at the end of the pipeline. `SORT` before `STATS` has no effect — aggregation destroys row order.

---

## Real Queries with Annotated Results

### Traffic by source country

```esql
FROM kibana_sample_data_logs
| STATS requests = COUNT(*) BY geo.src
| SORT requests DESC
| LIMIT 10
```

**Result:**
```
requests | geo.src
---------|--------
14074    | US
```

**Insight**: Every single request in this dataset originates from the US (`geo.src = "US"`). The destinations vary (China, India, US, Indonesia, Brazil...) but the source is always US. This is a key characteristic of the sample data — do not use it to test geo-filtering on source country.

---

### Traffic corridors — source to destination

```esql
FROM kibana_sample_data_logs
| STATS requests = COUNT(*), total_bytes = SUM(bytes)
    BY geo.srcdest
| SORT requests DESC
| LIMIT 10
```

**Real result:**
```
requests | total_bytes | geo.srcdest
---------|-------------|------------
2621     | 14,888,980  | US:CN
2323     | 12,974,807  | US:IN
1120     | 6,220,323   | US:US
487      | 2,701,863   | US:ID
411      | 2,395,461   | US:BR
318      | 1,768,678   | US:PK
315      | 1,822,394   | US:BD
304      | 1,753,428   | US:NG
259      | 1,456,748   | US:RU
248      | 1,402,564   | US:MX
```

**Insight**: China (CN) and India (IN) are the top download destinations, together accounting for ~35% of all traffic. US:US (domestic) is third. This likely represents Elasticsearch binary downloads from the APAC region.

---

### Top hosts by request volume

```esql
FROM kibana_sample_data_logs
| STATS requests = COUNT(*) BY host
| SORT requests DESC
| LIMIT 5
```

**Real result:**
```
requests | host
---------|-------------------------------
6488     | artifacts.elastic.co
4779     | www.elastic.co
2255     | cdn.elastic-elastic-elastic.org
552      | elastic-elastic-elastic.org
```

**Insight**: `artifacts.elastic.co` serves 46% of all traffic — this is the binary download host (`.zip`, `.deb`, `.rpm` files). `www.elastic.co` is the main website. The `cdn.elastic-elastic-elastic.org` subdomain handles CDN-routed content.

---

## Multi-Pipe Pipeline Patterns

### Pattern 1 — Aggregate → Compute → Filter → Sort

The most reusable pattern in ES|QL. Aggregate first, then compute derived metrics, then filter noise, then sort.

```esql
FROM kibana_sample_data_logs
| STATS
    total  = COUNT(*),
    errors = COUNT(*) WHERE response.keyword != "200"
  BY clientip
| EVAL error_rate = ROUND(100.0 * errors / total, 1)
| WHERE total > 10                    -- remove low-volume IPs (statistical noise)
| SORT error_rate DESC
| LIMIT 15
```

> **Why `WHERE total > 10`**: An IP with 1 request and 1 error shows a 100% error rate — technically true but meaningless. Filtering by minimum volume is standard practice in rate calculations.

---

### Pattern 2 — Two-dimension grouping

Group by multiple fields simultaneously to build a matrix view.

```esql
FROM kibana_sample_data_logs
| STATS total_mb = ROUND(SUM(bytes) / 1048576.0, 2)
    BY
      hour = DATE_TRUNC(1 hour, @timestamp),
      extension.keyword
| WHERE total_mb > 1
| SORT hour ASC, total_mb DESC
```

This produces a row per `hour + extension` combination — the raw data for a heatmap.

---

### Pattern 3 — EVAL before STATS

When your aggregation depends on a computed column, EVAL must come first.

```esql
FROM kibana_sample_data_logs
| EVAL hour    = DATE_TRUNC(1 hour, @timestamp)          -- compute bucket
| EVAL is_error = CASE(TO_INTEGER(response.keyword) >= 500, 1, 0)  -- flag errors
| STATS
    total     = COUNT(*),
    errors    = SUM(is_error),
    error_pct = ROUND(SUM(is_error) * 100.0 / COUNT(*), 2)
  BY hour
| SORT hour ASC
```

> **Order matters**: `EVAL` creates the column. `STATS` consumes it. If you swap them, ES|QL throws a "column not found" error.

---

## Common Mistakes and Fixes

| Mistake | Error or symptom | Fix |
|---|---|---|
| `WHERE response >= 500` on a text field | `verification_exception: first argument is [text]` | Use `TO_INTEGER(response.keyword) >= 500` inside EVAL |
| `SORT` before `STATS` | No effect — sort is destroyed by aggregation | Always STATS first, SORT last |
| `WHERE` on an EVAL column placed before EVAL | Column not found error | Move WHERE after the EVAL that creates the column |
| `WHERE bytes > 50000` returns 0 rows | Not an error — data has no rows matching | Check `MAX(bytes)` first before writing range filters |
| Aggregating without `BY` | Returns a single global row | Add `BY field` if you need per-group breakdown |
| Forgetting `LIMIT` | Returns up to 10,000 rows by default | Add `LIMIT` to cap and speed up results |
| Using `host` instead of `host.keyword` in STATS | Groups by analyzed text tokens, not full hostname | Always use `.keyword` sub-field for grouping/filtering |

---

## Key Numbers from This Dataset

| Metric | Value | How to verify |
|---|---|---|
| Total documents | 14,074 | `FROM kibana_sample_data_logs \| STATS COUNT(*)` |
| All traffic source | US only | `STATS COUNT(*) BY geo.src` |
| Top destination | US:CN (2,621 requests) | `STATS COUNT(*) BY geo.srcdest \| SORT DESC` |
| Error rate | 3.13% (441 docs) | `EVAL is_error \| STATS SUM(is_error) / COUNT(*)` |
| All errors are 503 | 441 × 503, 0 × 500 | `STATS COUNT(*) BY response.keyword` |
| Max response size | ~20KB | `STATS MAX(bytes)` |
| Most common size | Medium 1–10kb (85%) | `EVAL size_band \| STATS COUNT(*) BY size_band` |
| Top host | artifacts.elastic.co (6,488) | `STATS COUNT(*) BY host \| SORT DESC` |
| Days of data | ~30 days | `STATS MIN(@timestamp), MAX(@timestamp)` |
| Peak traffic hour | 10:00–12:00 UTC | `DATE_TRUNC(1 hour) \| STATS COUNT(*)` |

---

## Quick Decision Tree

```
What do I want to do?
│
├─ Narrow the time window?
│   └─ TIME PICKER (free, shard-level)
│
├─ Filter on a known indexed field?
│   └─ DSL FILTER BAR
│       ├─ Exact value         → "is"
│       ├─ Multiple values     → "is one of"
│       ├─ Range               → "is between"
│       └─ Field must exist    → "exists"
│
└─ Everything else?
    └─ ES|QL
        ├─ Compute a new column          → EVAL
        ├─ Filter on computed value      → WHERE (after EVAL)
        ├─ Group and summarise           → STATS ... BY
        ├─ Time buckets                  → DATE_TRUNC
        ├─ Extract time parts            → DATE_FORMAT
        ├─ Select / remove columns       → KEEP / DROP
        ├─ Rename columns                → RENAME
        ├─ Order results                 → SORT
        └─ Cap result count              → LIMIT
```

---

## Async Queries — for large or frozen datasets

When querying months of archived data, use the async API so the query runs in the background.

```json
POST /_query/async
{
  "query": """
    FROM logs-2025-*
    | WHERE response.keyword == "503"
    | STATS count = COUNT(*) BY host.keyword
    | SORT count DESC
  """,
  "wait_for_completion_timeout": "5s",
  "keep_on_completion": true
}
```

If `is_running: true` is returned, poll with:

```json
GET /_query/async/{id}?wait_for_completion_timeout=10s
```

When `is_running: false`, the results are ready.

---

## Parameters — safely injecting dynamic values

Never build queries by string concatenation. Use named parameters to prevent injection and improve reuse.

```json
POST /_query
{
  "query": "FROM kibana_sample_data_logs | WHERE response.keyword == ?code | STATS COUNT(*)",
  "params": [{"code": "503"}]
}
```

- `?name` — for values (strings, numbers, booleans)
- `??name` — for field names and function names (identifiers)

```json
{
  "query": "FROM kibana_sample_data_logs | STATS result = ??fn(??field) BY ??group",
  "params": [
    {"fn": "AVG"},
    {"field": "bytes"},
    {"group": "extension.keyword"}
  ]
}
```

> Never mix `?name` and `?1` positional styles in the same query.
